In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DVA_Project/data/processed/statistical_dataset.csv")

In [ ]:
df.shape

(1042384, 32)

In [ ]:
df_final = df.copy()

In [ ]:
df_final.shape

(1042384, 32)

##Handle Skewness (Keeping OUTLIERS)

In [ ]:
df_final['publish_time'] = pd.to_datetime(df_final['publish_time'])
df_final = df_final[df_final['publish_time'].dt.year >= 2018]


skew_cols = ['views', 'likes', 'comments']

for col in skew_cols:
    df_final[f'{col}_log'] = np.log1p(df_final[col])


#  Cap Extreme Values (for visualization only)


cap_cols = ['views', 'likes', 'comments']

for col in cap_cols:
    upper = df_final[col].quantile(0.99)
    df_final[f'{col}_capped'] = np.clip(df_final[col], None, upper)




In [ ]:
df_final.shape

(1042384, 38)

## Sampling (20k per country)

In [ ]:
df_final = df_final.sort_values('publish_time', ascending=False)

df_sampled = []

for country in df_final['country'].unique():
    temp = df_final[df_final['country'] == country]

    n = min(len(temp), 20000)

    recent_n = int(n * 0.6)
    random_n = n - recent_n

    recent = temp.head(recent_n)
    random = temp.sample(n=random_n, random_state=42)

    df_sampled.append(pd.concat([recent, random]))

df_final = pd.concat(df_sampled).drop_duplicates().reset_index(drop=True)

In [ ]:
df_final.shape

(78586, 38)

In [ ]:
print(df_final['country'].value_counts())

country
United States     19665
United Kingdom    19660
Canada            19660
India             19601
Name: count, dtype: int64


##Select Final Columns

In [ ]:

final_cols = [
    # identity
    'country', 'publish_time',

    # attention
    'views', 'views_log', 'views_capped',

    # engagement
    'likes', 'comments',
    'engagement_rate',
    'like_ratio', 'comment_ratio',

    # influence
    'virality_score',
    'attention', 'retention', 'influence',

    # content features
    'title_length',
    'title_uppercase_ratio',
    'title_has_exclamation',
    'tag_count', 'has_tags',

    # time features
    'publish_hour',
    'publish_day',
    'is_weekend',
    'time_to_trend'
]

df_final = df_final[final_cols]




In [ ]:
df_final.shape

(78586, 23)

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78586 entries, 0 to 78585
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   country                78586 non-null  object             
 1   publish_time           78586 non-null  datetime64[ns, UTC]
 2   views                  78586 non-null  int64              
 3   views_log              78586 non-null  float64            
 4   views_capped           78586 non-null  int64              
 5   likes                  78586 non-null  int64              
 6   comments               78586 non-null  int64              
 7   engagement_rate        78583 non-null  float64            
 8   like_ratio             78583 non-null  float64            
 9   comment_ratio          78579 non-null  float64            
 10  virality_score         78586 non-null  float64            
 11  attention              78586 non-null  int64          

##Export for Tableau

In [ ]:


df_final.to_csv("/content/drive/MyDrive/DVA_Project/data/processed/final_dataset_tableau.csv", index=False)

print("Final dataset shape:", df_final.shape)
print("Dataset ready for Tableau / Modeling.")

Final dataset shape: (78586, 23)
Dataset ready for Tableau / Modeling.
